# 6 · Verify artifacts and measure size

## Goal

Recompute SHA-256 for each distributable artifact, compare embedded payload metadata, and identify size drivers without modifying any image.

## Prerequisites

Complete the image build in chapter 5. The precomputed hashes are integrity measurements, **not release signatures**.

In [ ]:
from pathlib import Path
import subprocess, sys
START=Path.cwd().resolve()
ROOT=next((p for p in (START,*START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()),None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+"," ".join(map(str,argv)),flush=True)
    return subprocess.run([str(a) for a in argv],cwd=ROOT,check=True)
print("Source:",ROOT)

## Action 1 · Load build reports

In [ ]:
import json, hashlib
OUTPUT=ROOT/"build/output"
manifest=json.loads((OUTPUT/"artifacts.json").read_text())
footprint=json.loads((OUTPUT/"footprint.json").read_text())
payload=json.loads((OUTPUT/"payload.json").read_text())
print("Distributable artifacts:",", ".join(manifest))

## Action 2 · Recompute hashes using bounded memory

In [ ]:
def digest(path):
    h=hashlib.sha256()
    with path.open("rb") as src:
        for chunk in iter(lambda:src.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()
for name,record in manifest.items():
    file=OUTPUT/name
    assert file.stat().st_size==record["size"], f"Size mismatch: {name}"
    assert digest(file)==record["sha256"], f"Digest mismatch: {name}"
    print("Verified:",name)

## Action 3 · Compare the installer payload with the server image

In [ ]:
for key,name in (("raw","raw/sc-linux.raw"),("compressed","sc-linux.raw.zst")):
    file=OUTPUT/name
    assert payload[key]["size"]==file.stat().st_size
    assert payload[key]["sha256"]==digest(file)
    print("Payload matches:",name)

## Action 4 · Review sizes and largest packages

In [ ]:
for name,value in footprint["artifacts"].items():
    print(f"{name}: {value}")
print("\nLargest installed packages (KiB):")
for pkg in footprint["packages"]["largest"][:12]:
    print(f"{pkg['installed_size_kib']:>9,}  {pkg['package']}")

## Check

All artifact hashes, sizes and embedded payload descriptors must match. The RAW includes partition and filesystem overhead; package Installed-Size is an estimate. See [footprint methodology](../docs/footprint.md).

## Next

[Boot the actual images](07-boot.ipynb).